In [1]:
# !pip install convokit zstandard
# !pip uninstall -y numpy scikit-learn transformers
# !pip install numpy==1.24.4 scikit-learn==1.3.2 transformers==4.36.2
!nvidia-smi

Sat Apr 12 21:15:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4000               Off |   00000000:00:05.0 Off |                  Off |
| 41%   33C    P8             13W /  140W |       2MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from convokit import Corpus, download
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
from torch.optim import AdamW
import random
from datasets import load_dataset
from tqdm import tqdm
import random


In [3]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


special_tokens = ["<SPK1>", "<SPK2>"]
tokenizer.add_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

# Optional: average embedding init
with torch.no_grad():
    embedding = model.get_input_embeddings()
    avg = embedding.weight[:-len(special_tokens)].mean(dim=0)
    for tok in special_tokens:
        idx = tokenizer.convert_tokens_to_ids(tok)
        embedding.weight[idx] = avg


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Freeze everything except embeddings
for param in model.parameters():
    param.requires_grad = False
model.transformer.wte.weight.requires_grad = True

optimizer = AdamW(model.parameters(), lr=1e-4)


In [5]:
corpus = Corpus(filename=download("switchboard-processed-corpus"))

dialogs = []
for convo in tqdm(corpus.iter_conversations(), desc="Parsing conversations"):
    utterances = list(convo.iter_utterances())
    if len(utterances) < 2:
        continue

    speakers = list({utt.speaker.id for utt in utterances})
    if len(speakers) != 2:
        continue
    
    if random.random() > 0.5:
        speaker_map = {speakers[0]: "<SPK1>", speakers[1]: "<SPK2>"}
    else:
        speaker_map = {speakers[0]: "<SPK2>", speakers[1]: "<SPK1>"}
    
    dialog = []

    for utt in utterances:
        text = utt.text.strip().replace("\n", " ")
        if text:
            dialog.append(f"{speaker_map[utt.speaker.id]} {text}")

    if dialog:
        dialogs.append(" ".join(dialog))


################################

dataset = load_dataset("daily_dialog", trust_remote_code=True)

for sample in tqdm(dataset["train"], desc="Parsing DailyDialog"):
    utterances = sample["dialog"]
    if len(utterances) < 2:
        continue

    dialog = []
    flip = random.random() > 0.5  # randomly assign SPK1/SPK2
    for i, text in enumerate(utterances):
        spk = "<SPK1>" if (i % 2 == 0) ^ flip else "<SPK2>"
        dialog.append(f"{spk} {text.strip()}")

    dialogs.append(" ".join(dialog))

###########################################

dataset = load_dataset("agentlans/Conversational-Reasoning-Topical-Chat")

for sample in tqdm(dataset["train"], desc="Parsing TopicalChat"):
    dialog = []
    turns = sample['conversations'][1:]

    # Skip single-utterance examples
    if len(turns) < 2:
        continue

    # Identify both speakers and assign SPK1/SPK2 randomly
    all_speakers = list(set(turn["from"] for turn in turns))
    if len(all_speakers) != 2:
        continue

    if random.random() > 0.5:
        speaker_map = {all_speakers[0]: "<SPK1>", all_speakers[1]: "<SPK2>"}
    else:
        speaker_map = {all_speakers[0]: "<SPK2>", all_speakers[1]: "<SPK1>"}

    for turn in turns:
        text = turn["value"].strip().replace("\n", " ")
        if text:
            dialog.append(f"{speaker_map[turn['from']]} {text}")

    if dialog:
        dialogs.append(" ".join(dialog))

class DialogDataset(Dataset):
    def __init__(self, dialogs, tokenizer, max_length=512):
        self.inputs = []
        for dialog in tqdm(dialogs, desc="Tokenizing dialogs"):
            enc = tokenizer(dialog, truncation=True, max_length=max_length, return_tensors="pt")
            self.inputs.append(enc.input_ids.squeeze(0))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        ids = self.inputs[idx]
        return ids, ids  # input and label are same for LM

dataset = DialogDataset(dialogs, tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda x: tokenizer.pad(
    [{"input_ids": i[0]} for i in x],
    return_tensors="pt",
    padding=True
))


Dataset already exists at /root/.convokit/saved-corpora/switchboard-processed-corpus


Parsing conversations: 1155it [00:00, 16106.46it/s]
Tokenizing dialogs: 100%|██████████| 20901/20901 [00:24<00:00, 863.87it/s] 


In [6]:
print(dialogs[0])

<SPK2> What kind of experience do you , do you have , then with child care ? <SPK1> I think , uh , I wonder if that worked . <SPK2> Does it say something ? <SPK1> I think it usually does . You might try , uh , I do n't know , hold it down a little longer , and see if it , uh , - <SPK2> Okay pause > > Well , Does it usually make a recording or s- , <SPK1> Well , I do n't remember . It seemed like it did , but it might not . I guess -- -- I guess we can start . Uh , No , I do n't , I do n't have any kids . I , uh , my sister has a , she just had a baby , he 's about five months old and she was worrying about going back to work and what she was going to do with him and -- -- the different , - do you have kids ? <SPK2> I have three . Yeah , I do Yes , uh , I do n't work , though , but I used to work and , when I had two children . I work off and on just temporarily and usually find friends to babysit , but I do n't envy anybody who 's in that situation to find day care . But , does your si

In [8]:
model.train()
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    loop = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        labels = input_ids.clone()

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} complete — Average Loss: {avg_loss:.4f}")
    model.save_pretrained("fine-tuned-gpt2-speakers-6")
    tokenizer.save_pretrained("fine-tuned-gpt2-speakers-6")


Epoch 1: 100%|██████████| 2613/2613 [21:08<00:00,  2.06it/s, loss=2.42] 


Epoch 1 complete — Average Loss: 1.6202


Epoch 2: 100%|██████████| 2613/2613 [21:13<00:00,  2.05it/s, loss=1.59] 


Epoch 2 complete — Average Loss: 1.6007


Epoch 3: 100%|██████████| 2613/2613 [21:08<00:00,  2.06it/s, loss=1.48] 


...

Epoch 8 complete — Average Loss: 1.5323


Epoch 9: 100%|██████████| 2613/2613 [21:12<00:00,  2.05it/s, loss=1.65] 


Epoch 9 complete — Average Loss: 1.5256


Epoch 10: 100%|██████████| 2613/2613 [21:14<00:00,  2.05it/s, loss=0.879]


Epoch 10 complete — Average Loss: 1.5151
